# Chapter 3 — Description Logics
### Notebook 4 · Exercises

*Book reference: Section 3.4*

The book's exercises, executable. Assertions are the marking scheme.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the course root (the folder holding agentlab/) and this chapter importable.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "agentlab").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import agentlab
print(json.dumps(agentlab.describe_environment(), indent=1))

In [ ]:
import logging
import ch03_toolkit as dl
import pandas as pd
A = dl.Atomic
logging.getLogger("dspy").setLevel(logging.WARNING)
from agentlab.grading import Grader
CHECK = Grader('Chapter 3 · Notebook 4 exercises')

### Exercise R1 — Translate English into DL

Express each statement as a DL axiom and verify the intended entailment holds.

In [ ]:
t_r1 = dl.TBox()
# TODO: 'A vegetarian is a person that eats no meat.'
#       'A vegan is a vegetarian that eats no dairy.'
#       'Steak is meat.'
# Use the class names Vegetarian, Person, Meat, Vegan, Dairy, Steak and the role eats.

In [ ]:
# Check for Exercise R1 -- tests the names your code defines.
with CHECK.check('R1', points=1):
    assert dl.subsumes(A('Vegan'), A('Vegetarian'), t_r1)
    assert not dl.subsumes(A('Vegetarian'), A('Vegan'), t_r1)
    assert not dl.satisfiable(dl.And(A('Vegetarian'), dl.Exists('eats', A('Steak'))), t_r1).satisfiable
    assert dl.satisfiable(A('Vegan'), t_r1).satisfiable

### Exercise R2 — Name the logic for five knowledge bases

Report the DL for each, and rank them by worst-case reasoning complexity.

In [ ]:
builders = {
    'plain':      lambda t: t.add(A('A'), A('B')),
    'negation':   lambda t: t.add(A('A'), dl.Not(A('B'))),
    'transitive': lambda t: (t.add(A('A'), dl.Exists('r', A('B'))),
                             t.transitive_roles.add('r')),
    'inverse+num': lambda t: (t.add(A('A'), dl.Exists(dl.Inverse('r'), A('B'))),
                              t.add(A('B'), dl.AtLeast(2, 'r', A('C')))),
    'everything': lambda t: (t.add(A('A'), dl.Exists(dl.Inverse('r'), A('B'))),
                             t.add(A('B'), dl.AtLeast(2, 'r', A('C'))),
                             t.transitive_roles.add('r'),
                             t.role_hierarchy.append(('r', 's')),
                             t.nominals.add('bob')),
}
names_r2 = {}   # TODO: kb -> its DL name

In [ ]:
# Check for Exercise R2 -- tests the names your code defines.
with CHECK.check('R2', points=1):
    assert names_r2 == {'plain': 'ALC', 'negation': 'ALC', 'transitive': 'S',
                        'inverse+num': 'ALCIQ', 'everything': 'SHOIQ'}

### Exercise R3 — Explain an unsatisfiability to a colleague

Take the unsatisfiable `Giraffe ⊓ Carnivore`, and produce the shortest chain of axioms that explains it — the *justification*, in DL terms.

> **Hint.** Search subsets of the axioms from smallest upward, keeping the first that still makes the concept unsatisfiable.

In [ ]:
w = dl.wildlife_tbox()
target = dl.And(A('Giraffe'), A('Carnivore'))
justification = None   # TODO: a smallest subset of w.axioms that still makes target unsatisfiable

In [ ]:
# Check for Exercise R3 -- tests the names your code defines.
with CHECK.check('R3', points=1):
    just = list(justification)
    assert all(ax in w.axioms for ax in just)
    assert not dl.satisfiable(target, dl.TBox(just)).satisfiable, "the subset must still entail it"
    assert len(just) == 4, f"{len(just)} axioms: the smallest justification has 4"

## Where this leaves you

You have a reasoner, you can name the logic you are in, and you can explain an unsatisfiability with a minimal justification. The problem set (`05_assignment`) puts Claude next to that reasoner and asks the questions a practising engineer actually faces: when should the verdict come from the model and when from the reasoner, what does the model still contribute once a sound checker exists, and is a model that scores well on the Pizza ontology reasoning — or remembering?

### Exercise checks

Each exercise's check tests the names *your* code defines; a failing check is recorded, not raised, so the notebook runs to the end. The worked solutions are in the `_solutions` notebook next to this one.

In [ ]:
_ = CHECK.summary(['R1', 'R2', 'R3'])